In [ ]:
# ============================================
# K-MEDIAS SOBRE T_final_unsupervised
# - Carga CSV
# - Ajusta KMeans (k-means++) para un rango de K.
# - Guarda:
#   * elbow.png           (método del codo)
#   * silhouette.png      (silueta media por K)
#   * kmeans_k{K}.png     (PC1-PC2 con partición y centroides)
#   * T_final_unsupervised_con_clusters.csv (features + label_k{K})
# ============================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from matplotlib.colors import ListedColormap
from scipy.spatial import Voronoi

In [ ]:
# -------- CONFIG --------
X = pd.read_csv("T_final_unsupervised.csv")
############################################
############################################
K_RANGE     = list(range(2, 11))           # para codo/silueta
############################################
############################################
RANDOM_SEED = 7
OUTDIR      = Path("kmeans_out"); OUTDIR.mkdir(exist_ok=True)

In [ ]:
# -------- FUNCIONES AUX --------
def inertia_curve(X, ks, seed=RANDOM_SEED):
    vals = []
    for k in ks:
        km = KMeans(n_clusters=k, init="k-means++", n_init=20, max_iter=300,
                    random_state=seed, algorithm="lloyd")
        km.fit(X)
        vals.append(km.inertia_)
    return np.array(vals)

def silhouette_curve(X, ks, seed=RANDOM_SEED):
    vals = []
    for k in ks:
        km = KMeans(n_clusters=k, init="k-means++", n_init=20, max_iter=300,
                    random_state=seed, algorithm="lloyd")
        labels = km.fit_predict(X)
        vals.append(silhouette_score(X, labels))
    return np.array(vals)

def plot_elbow(ks, inertias, path):
    plt.figure(figsize=(7.2, 5.0))
    plt.plot(ks, inertias, marker="o")
    plt.xticks(ks)
    plt.xlabel("K (número de clusters)")
    plt.ylabel("Inercia (SSE intra-cluster)")
    plt.title("Método del codo")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close()

def plot_silhouette(ks, sils, path):
    plt.figure(figsize=(7.2, 5.0))
    plt.plot(ks, sils, marker="o")
    plt.xticks(ks)
    plt.xlabel("K (número de clusters)")
    plt.ylabel("Silueta media")
    plt.title("Coeficiente de silueta por K")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close()

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi

def plot_partition_pc12_voronoi(X, labels, centers, path, h=0.01, alpha=0.20):
    """
    Dibuja las regiones de decisión de K-medias en el plano PC1–PC2 (o primeras
    dos columnas de X) coloreando por el centroide más cercano. Guarda y muestra.
    """
    # --- Plano a graficar ---
    cols = list(X.columns)
    if {"PC1", "PC2"}.issubset(cols):
        x1, x2 = X["PC1"].to_numpy(), X["PC2"].to_numpy()
        C = centers[:, [cols.index("PC1"), cols.index("PC2")]]
        xlabel, ylabel = "PC1", "PC2"
    else:
        x1, x2 = X.iloc[:, 0].to_numpy(), X.iloc[:, 1].to_numpy()
        C = centers[:, :2]
        xlabel, ylabel = cols[0], cols[1]

    # --- Colores consistentes ---
    k = len(np.unique(labels))
    cmap = plt.cm.get_cmap("viridis", k)

    # --- Malla fina para pintar regiones completas ---
    pad = 0.6
    x1min, x1max = x1.min() - pad, x1.max() + pad
    x2min, x2max = x2.min() - pad, x2.max() + pad
    xx, yy = np.meshgrid(np.arange(x1min, x1max, h),
                         np.arange(x2min, x2max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # Asignación por centroide más cercano (sin usar el objeto KMeans)
    d2 = ((grid[:, None, :] - C[None, :, :])**2).sum(axis=2)   # dist^2 a cada centroide
    Z = d2.argmin(axis=1).reshape(xx.shape)

    # --- Plot limpio (regiones + contornos + puntos + centroides) ---
    plt.figure(figsize=(8.5, 6.0))
    plt.pcolormesh(xx, yy, Z, cmap=cmap, shading="nearest", alpha=alpha)
    plt.contour(xx, yy, Z, levels=np.arange(k)-0.5, colors="k", linewidths=0.6, alpha=0.45)

    plt.scatter(x1, x2, c=cmap(labels), s=28, marker="x")
    plt.scatter(C[:, 0], C[:, 1], c="red", s=180, marker="X")

    plt.xlim(x1min, x1max); plt.ylim(x2min, x2max)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title("K-medias: partición en PC1–PC2")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show(); plt.close()

# === Silueta detallada por cluster (perfil) ===
from sklearn.metrics import silhouette_samples, silhouette_score

def plot_silhouette_profile(X, labels, path, title_prefix="Diagrama de silueta"):
    K = len(np.unique(labels))
    cmap = plt.cm.get_cmap("viridis", K)

    sil_samples = silhouette_samples(X, labels)
    sil_avg     = float(np.mean(sil_samples))

    plt.figure(figsize=(9.0, 5.0))
    y_lower = 10  # separador vertical

    for k in range(K):
        vals = sil_samples[labels == k]
        vals.sort()
        n_k = len(vals)
        y_upper = y_lower + n_k

        plt.fill_betweenx(
            np.arange(y_lower, y_upper),
            0, vals,
            facecolor=cmap(k), edgecolor="white", linewidth=0.8, alpha=0.95
        )
        plt.text(0.0, y_lower + 0.5 * n_k, f"Cluster {k}", va="center")

        y_lower = y_upper + 10  # espacio entre clusters

    # línea de la silueta media
    plt.axvline(sil_avg, color="tab:orange", linestyle="--", linewidth=1.8)
    plt.xlabel("Valor de silueta")
    plt.ylabel("Muestras por cluster (apiladas)")
    plt.title(f"{title_prefix} (K={K}) — Silueta media = {sil_avg:.3f}")
    plt.xlim(-0.1, 1.0)
    plt.yticks([])
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show(); plt.close()

In [ ]:
# -------- CODO Y SILUETA --------
inertias  = inertia_curve(X, K_RANGE, seed=RANDOM_SEED)
silhouettes = silhouette_curve(X, K_RANGE, seed=RANDOM_SEED)

plot_elbow(K_RANGE, inertias, OUTDIR / "elbow.png")
plot_silhouette(K_RANGE, silhouettes, OUTDIR / "silhouette.png")

In [ ]:
# -------- AJUSTE FINAL Y EXPORT --------

#################################################
#################################################
K_FINAL = 5
#################################################
#################################################

kmeans = KMeans(n_clusters = K_FINAL, init="k-means++", n_init=50, max_iter=300, random_state=RANDOM_SEED, algorithm="lloyd")
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_
centers

In [ ]:
# Figura PC1–PC2 con centroids
plot_partition_pc12_voronoi(X, labels, centers, OUTDIR / f"kmeans_k{K_FINAL}.png")

In [ ]:
# Llamada (usa los 'labels' ya calculados para K_FINAL):
plot_silhouette_profile(X, labels, OUTDIR / f"silhouette_profile_k{K_FINAL}.png")

metrics = {
    "k_final": int(K_FINAL),
    "inertia_final": float(kmeans.inertia_),
    "silhouette_final": float(silhouette_score(X, labels)),
    "cluster_sizes": {int(k): int(v) for k, v in pd.Series(labels).value_counts().sort_index().to_dict().items()},
}

print("\nK_FINAL =", K_FINAL)
print("Inercia final:", metrics["inertia_final"])
print("Silueta final:", metrics["silhouette_final"])
print("Tamaños por cluster:", metrics["cluster_sizes"])

In [ ]:
# -------- GUARDADOS (OBJETOS Y MÉTRICAS) --------

import os, zipfile
# Dataset con etiquetas
out_df = X.copy()
out_df[f"label_k{K_FINAL}"] = labels
out_df_path = OUTDIR / "T_final_unsupervised_con_clusters.csv"
out_df.to_csv(out_df_path, index=False)

# Centros y etiquetas (tablas auxiliares)
pd.DataFrame(centers, columns=X.columns).to_csv(OUTDIR / f"centers_k{K_FINAL}.csv", index=False)
pd.DataFrame({"label": labels}).to_csv(OUTDIR / f"labels_k{K_FINAL}.csv", index=False)

# Modelo KMeans (joblib)
try:
    import joblib
    joblib.dump(kmeans, OUTDIR / f"kmeans_k{K_FINAL}.joblib")
except Exception as e:
    with open(OUTDIR / "joblib_error.txt", "w", encoding="utf-8") as f:
        f.write(str(e))

# configuración en JSON
with open(OUTDIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

from pathlib import Path

config = {
    "input_csv": "T_final_unsupervised.csv",           # <- cadena, no DataFrame
    "k_range": list(map(int, K_RANGE)),
    "k_final": int(K_FINAL),
    "random_seed": int(RANDOM_SEED),
    "n_samples": int(len(X)),
    "n_features": int(X.shape[1]),
    "columns_used": X.columns.tolist()                 # <- lista serializable (opcional)
}
with open(OUTDIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


# -------- CREAR ZIP (BUNDLE) --------
bundle_name = f"kmeans_k{K_FINAL}_bundle.zip"
zip_path = OUTDIR / bundle_name

to_zip = [
    OUTDIR / "elbow.png",
    OUTDIR / "silhouette.png",
    OUTDIR / f"kmeans_k{K_FINAL}.png",
    OUTDIR / "elbow_curve.csv",
    OUTDIR / "silhouette_curve.csv",
    OUTDIR / "T_final_unsupervised_con_clusters.csv",
    OUTDIR / f"centers_k{K_FINAL}.csv",
    OUTDIR / f"labels_k{K_FINAL}.csv",
    OUTDIR / f"kmeans_k{K_FINAL}.joblib",
    OUTDIR / "metrics.json",
    OUTDIR / "config.json",
]


with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in to_zip:
        if isinstance(f, Path): f = f
        if os.path.exists(f):
            zf.write(f, arcname=Path(f).name)

print("ZIP:", zip_path.resolve())